# 03. Data Integration & Momentum Scoring

This notebook merges the cleaned internal data with the freshly extracted external web data. Crucially, we merge strictly on the validated `restaurant_id`.

Finally, it calculates the **Momentum** and **Stability** scores using standardized scaling (Min-Max) instead of the flawed Maximum normalization, preventing outliers from crushing the scores of 'Hidden Gems'.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

## 1. Load Data

In [ ]:
CLEAN_FILE = "Processed/master_cleaned_data.csv"
SCRAPE_FILE = "Processed/master_web_scraping_data.csv"
OUTPUT_FILE = "Processed/master_all_data.csv"

print("Loading Data...")
try:
    df_master = pd.read_csv(CLEAN_FILE)
    print(f"Loaded {len(df_master)} master records.")
except Exception as e:
    print(f"Failed to load cleaned data: {e}")
    df_master = pd.DataFrame()

try:
    df_scrape = pd.read_csv(SCRAPE_FILE)
    print(f"Loaded {len(df_scrape)} scraped records.")
except Exception as e:
    print(f"Failed to load scraped data: {e}")
    df_scrape = pd.DataFrame()

## 2. Direct Validation Merge

Joining on `restaurant_id` provides guaranteed mapping, solving the string loss previously seen.

In [ ]:
if not df_master.empty and not df_scrape.empty and 'restaurant_id' in df_scrape.columns:
    # Direct ID merge guarantee 100% accuracy of scraped rows
    cols_to_drop = ['restaurant_name_en', 'restaurant_name_th', 'extracted_location']
    df_merged = pd.merge(df_master, df_scrape.drop(columns=cols_to_drop, errors='ignore'), on='restaurant_id', how='left')
    print(f"Merged shape: {df_merged.shape}")
else:
    print("Warning: ID column missing or dataframes empty. Skipping merge.")
    df_merged = df_master.copy()

# Ensure numeric columns are filled
scrape_cols = [
    'gmaps_rating', 'gmaps_reviews', 
    'tiktok_views', 'tiktok_likes', 'tiktok_shares',
    'wongnai_rating', 'wongnai_reviews',
    'facebook_likes'
]
for c in scrape_cols:
    if c in df_merged.columns:
        df_merged[c] = df_merged[c].fillna(0)
    else:
        df_merged[c] = 0


## 3. Prioritization Scoring Logic

We use **Min-Max Scaling** `(x - min) / (max - min)` across specific cohorts to normalize scores evenly, rather than doing completely un-standardized division by the mathematical maximum `(x / max)`. We apply `np.log1p()` for highly skewed engagement metrics before scaling.

In [ ]:
def min_max_scale(series):
    """Scales a Pandas Series between 0 and 100"""
    s_min = series.min()
    s_max = series.max()
    if s_max - s_min == 0:
        return pd.Series(0, index=series.index)
    return ((series - s_min) / (s_max - s_min)) * 100

# Calculate raw engagement using TikTok Data + Facebook Data + internal KOLs
df_merged['raw_momentum'] = (
    (df_merged['kol_views'] + df_merged['tiktok_views']) * 0.4 + 
    (df_merged['kol_likes'] + df_merged['tiktok_likes']) * 1.5 + 
    (df_merged['facebook_likes']) * 0.1
)

# Normalize Momentum (Log Transform to handle virality spikes, then Min-Max)
df_merged['momentum_score'] = min_max_scale(np.log1p(df_merged['raw_momentum']))

# Calculate Internal Stability (Revenue & Bookings)
internal_raw = np.log1p(df_merged['total_revenue']) * 0.7 + np.log1p(df_merged['total_bookings']) * 0.3
df_merged['internal_score'] = min_max_scale(internal_raw)

# Calculate External Stability (Google Maps + Wongnai)
# Log reduces weight of 10K review anomalies
df_merged['reliability_factor'] = np.log1p(df_merged['gmaps_reviews'] + df_merged['wongnai_reviews']) 
external_raw = ((df_merged['gmaps_rating'] * 0.7) + (df_merged['wongnai_rating'] * 0.3)) * df_merged['reliability_factor']
df_merged['external_quality_score'] = min_max_scale(external_raw)

# Final Stability = 50% Internal + 50% External
df_merged['stability_score'] = (df_merged['internal_score'] * 0.5) + (df_merged['external_quality_score'] * 0.5)

# Final Priority = 70% Momentum (We want Trending) + 30% Stability
df_merged['final_priority_score'] = (df_merged['momentum_score'] * 0.7) + (df_merged['stability_score'] * 0.3)

## 4. Categorize & Validate

In [ ]:
df_merged['validation_status'] = 'Unknown'

# Validated if there is scraping data
scraped_mask = (df_merged['gmaps_reviews'] > 0) | (df_merged['facebook_likes'] > 0) | (df_merged['tiktok_views'] > 0)
df_merged.loc[scraped_mask, 'validation_status'] = 'Verified'

# Viral Hit: High Momentum
df_merged.loc[(df_merged['momentum_score'] > 60), 'validation_status'] = 'Viral Hit'

# Hidden Gem: High External Quality BUT Low Internal Performance 
df_merged.loc[
    (df_merged['external_quality_score'] > 60) & 
    (df_merged['internal_score'] < 30), 
    'validation_status'
] = 'Hidden Gem'

# Cash Cow: High Internal Performance and Stable External Quality
df_merged.loc[(df_merged['internal_score'] > 80), 'validation_status'] = 'Cash Cow'

# Sort and Output
df_final = df_merged.sort_values('final_priority_score', ascending=False)
df_final.to_csv(OUTPUT_FILE, index=False)

print(f"\n✅ Merging and Scoring complete. Saved {len(df_final)} records to {OUTPUT_FILE}")
print("\n--- Top 10 Prioritized Restaurants ---")
display(df_final[['restaurant_name_en', 'final_priority_score', 'momentum_score', 'stability_score', 'validation_status']].head(10))